# 19 VaR与CVaR：风险价值度量

> **模块 2.3 — 风控与优化** | 理论学习 2h · 实战 3h

本 Notebook 系统讲解量化风控中最核心的两个指标：**VaR（Value at Risk，在险价值）**和**CVaR（Conditional VaR，条件在险价值）**，并用三种方法对一个真实的股票组合进行计算和对比。

## 🎯 学习目标

1. **理解 VaR 的定义**：在给定置信水平下，投资组合在特定时间内的最大可能损失
2. **三种 VaR 计算方法**：历史模拟法、参数法（方差-协方差法）、蒙特卡洛模拟法
3. **CVaR（Expected Shortfall）**：VaR 的补充指标，衡量超过 VaR 的尾部损失的期望值
4. **手算验证**：用简单数据手算 VaR，验证代码正确性
5. **回测检验**：Kupiec 检验评估 VaR 模型的准确性

## 📦 环境依赖

```python
numpy, pandas, matplotlib, scipy, yfinance
```

> ⚠️ 数据获取部分包含网络请求容错：如果 `yfinance` 不可用，自动降级为模拟数据。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("✓ 环境就绪")

## 1. VaR 概念介绍

**VaR（Value at Risk，在险价值）**回答一个问题：

> "在给定置信水平 $\alpha$ 下，投资组合在未来 $T$ 天内的最大可能损失是多少？"

$$P(\text{损失} > \text{VaR}_\alpha) = 1 - \alpha$$

- **95% VaR**：有 95% 的把握，损失不会超过 VaR
- **99% VaR**：有 99% 的把握，损失不会超过 VaR（更保守）

### 三种计算方法

| 方法 | 思路 | 优点 | 缺点 |
|------|------|------|------|
| **历史模拟法** | 用历史收益率分布的分位数 | 不假设分布，简单直观 | 依赖历史数据 |
| **参数法** | 假设收益率服从正态分布，$\text{VaR} = \mu + z_\alpha \sigma$ | 计算快 | 正态假设常不成立（肥尾） |
| **蒙特卡洛法** | 模拟大量随机情景，取分位数 | 灵活，可处理复杂组合 | 计算量大 |

## 2. 构建投资组合并获取数据

选择 5 只 A 股龙头构建等权组合，获取近 2 年日收益率数据。

> 如果网络不可用，自动使用模拟数据（带肥尾特征的 t 分布 + 随机相关性）。

In [ ]:
# ========== 数据获取（带降级） ==========
def get_stock_data():
    try:
        import yfinance as yf
        tickers = ['600519.SS', '000858.SZ', '601318.SS', '600036.SS', '300750.SZ']
        names = ['贵州茅台', '五粮液', '中国平安', '招商银行', '宁德时代']
        data = pd.DataFrame()
        for t in tickers:
            df = yf.download(t, start='2022-01-01', progress=False)
            if df.empty:
                raise Exception(f"{t} 数据为空")
            data[t] = df['Adj Close']
        returns = data.pct_change().dropna()
        print("✓ 使用真实数据 (yfinance)")
        return returns, names
    except Exception as e:
        print(f"yfinance 不可用: {e}")
        print("→ 降级为模拟数据...")
        np.random.seed(42)
        n_days, n_assets = 500, 5
        names = ['模拟资产A', '模拟资产B', '模拟资产C', '模拟资产D', '模拟资产E']
        returns = pd.DataFrame(np.random.standard_t(df=4, size=(n_days, n_assets)) * 0.015,
                               columns=[f'Asset_{i}' for i in range(n_assets)])
        corr = np.array([[1.0, 0.6, 0.5, 0.4, 0.3],[0.6, 1.0, 0.4, 0.3, 0.3],[0.5, 0.4, 1.0, 0.5, 0.2],[0.4, 0.3, 0.5, 1.0, 0.2],[0.3, 0.3, 0.2, 0.2, 1.0]])
        L = np.linalg.cholesky(corr)
        returns = returns @ L.T
        returns.columns = [f'Asset_{i}' for i in range(n_assets)]
        print("✓ 使用模拟数据 (t分布 + Cholesky相关性)")
        return returns, names

returns, stock_names = get_stock_data()
returns.index = pd.date_range(end='2024-12-31', periods=len(returns), freq='B')
print(f"\n数据概览：{len(returns)}个交易日, {returns.shape[1]}只股票, 覆盖约{len(returns)/252:.1f}年")
display(returns.head())
print("\n日收益率统计：")
display(returns.describe().round(4))

weights = np.array([0.2]*5)
portfolio_returns = returns @ weights
print(f"\n组合年化收益率: {portfolio_returns.mean()*252:.2%}")
print(f"组合年化波动率: {portfolio_returns.std()*np.sqrt(252):.2%}")

## 3. 方法一：历史模拟法（Historical Simulation）

### 原理

直接用历史收益率分布的分位数作为 VaR。不假设任何分布，是最直观的方法。

$$\text{VaR}_\alpha = -\text{Percentile}(r_1, r_2, ..., r_T, 1-\alpha)$$

In [ ]:
# ========== 历史模拟法 ==========
def var_historical(returns, confidence=0.95):
    return -np.percentile(returns, 100*(1-confidence))

def cvar_historical(returns, confidence=0.95):
    var = var_historical(returns, confidence)
    exceedances = returns[returns <= -var]
    return -exceedances.mean() if len(exceedances) > 0 else var

var_95_hs = var_historical(portfolio_returns, 0.95)
var_99_hs = var_historical(portfolio_returns, 0.99)

print("="*50)
print("📊 历史模拟法结果")
print("="*50)
print(f"95% VaR (日): {var_95_hs:.4f} = {var_95_hs:.2%}")
print(f"99% VaR (日): {var_99_hs:.4f} = {var_99_hs:.2%}")

# 手算验证
print("\n🔍 手算验证")
sorted_ret = np.sort(portfolio_returns)
n = len(sorted_ret)
idx_95 = int(n * 0.05)
manual_var = -sorted_ret[idx_95]
print(f"手算95% VaR: {manual_var:.6f} vs 代码: {var_95_hs:.6f}  {'✓ 一致' if np.isclose(manual_var, var_95_hs, rtol=0.01) else '⚠ 有差异'}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(portfolio_returns, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(-var_95_hs, color='red', linestyle='--', linewidth=2, label=f'95% VaR={var_95_hs:.2%}')
axes[0].axvline(-var_99_hs, color='darkred', linestyle='-', linewidth=2, label=f'99% VaR={var_99_hs:.2%}')
axes[0].set_xlabel('日收益率'); axes[0].set_ylabel('频率密度')
axes[0].set_title('组合日收益率分布 (历史模拟法)'); axes[0].legend()

cum_ret = (1+portfolio_returns).cumprod()
rolling_max = cum_ret.expanding().max()
drawdown = (cum_ret - rolling_max) / rolling_max
axes[1].plot(cum_ret.index, cum_ret, color='steelblue', linewidth=1.5, label='累计收益')
axes[1].fill_between(cum_ret.index, cum_ret, rolling_max, alpha=0.3, color='red', label='回撤')
axes[1].set_ylabel('累计净值'); axes[1].set_title('组合净值曲线与回撤'); axes[1].legend()
plt.tight_layout(); plt.savefig('var_historical.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. 方法二：参数法（方差-协方差法）

### 原理

假设组合收益率服从正态分布 $\mathcal{N}(\mu, \sigma^2)$：

$$\text{VaR}_\alpha = -(\mu + z_{1-\alpha} \cdot \sigma)$$

| 置信水平 | $z$ 值 |
|----------|--------|
| 95% | -1.645 |
| 99% | -2.326 |

- ✅ 计算极快，只需均值和标准差
- ❌ 正态假设在金融市场常不成立（**肥尾**现象）

In [ ]:
# ========== 参数法 ==========
def var_parametric(returns, confidence=0.95):
    mu, sigma = returns.mean(), returns.std()
    z = stats.norm.ppf(1-confidence)
    return -(mu + z*sigma), mu, sigma

def cvar_parametric(returns, confidence=0.95):
    mu, sigma = returns.mean(), returns.std()
    z = stats.norm.ppf(1-confidence)
    return -(mu - sigma*stats.norm.pdf(z)/(1-confidence))

var_95_par, mu, sigma = var_parametric(portfolio_returns, 0.95)
var_99_par, _, _ = var_parametric(portfolio_returns, 0.99)

print("="*50)
print("📊 参数法结果（假设正态分布）")
print("="*50)
print(f"日均收益率 μ: {mu:.6f}, 日波动率 σ: {sigma:.6f}")
print(f"z(95%): {stats.norm.ppf(0.05):.4f}, z(99%): {stats.norm.ppf(0.01):.4f}")
print(f"\n95% VaR: {var_95_par:.4f} = {var_95_par:.2%}")
print(f"99% VaR: {var_99_par:.4f} = {var_99_par:.2%}")

# 手算验证
manual_var_par = -(mu + (-1.645)*sigma)
print(f"\n🔍 手算验证: {manual_var_par:.6f} vs 代码: {var_95_par:.6f}  {'✓ 一致' if np.isclose(manual_var_par, var_95_par, rtol=0.01) else '⚠'}")

# 正态性检验
stat_jb, p_jb = stats.jarque_bera(portfolio_returns)
print(f"\n📊 正态性检验: Jarque-Bera p值={p_jb:.4f}")
print(f"偏度={stats.skew(portfolio_returns):.4f}, 峰度={stats.kurtosis(portfolio_returns):.4f} (正态=0, 正=肥尾)")
print(f"→ {'拒绝正态分布假设' if p_jb < 0.05 else '不能拒绝正态分布假设'}")

# QQ图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
stats.probplot(portfolio_returns, dist="norm", plot=axes[0])
axes[0].set_title('Q-Q图：组合收益率 vs 正态分布')
x = np.linspace(portfolio_returns.min(), portfolio_returns.max(), 200)
axes[1].hist(portfolio_returns, bins=50, density=True, alpha=0.7, color='steelblue', label='实际分布')
axes[1].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'正态拟合 N({mu:.4f},{sigma:.4f})')
axes[1].axvline(-var_95_par, color='orange', linestyle='--', linewidth=2, label=f'参数法95%VaR={var_95_par:.2%}')
axes[1].set_xlabel('日收益率'); axes[1].set_ylabel('频率密度')
axes[1].set_title('实际分布 vs 正态分布拟合'); axes[1].legend()
plt.tight_layout(); plt.savefig('var_parametric.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. 方法三：蒙特卡洛模拟法

### 原理

1. 估计组合收益率的均值和协方差矩阵
2. 生成大量（如 20000 次）多元正态随机收益率情景
3. 取模拟收益率分布的分位数作为 VaR

$$\mathbf{r}_{sim} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma}), \quad r_{p, sim} = \mathbf{w}^T \mathbf{r}_{sim}$$

- ✅ 灵活，可处理任意分布假设和复杂组合
- ❌ 计算量大，结果依赖模型假设

In [ ]:
# ========== 蒙特卡洛模拟法 ==========
def var_monte_carlo(returns, weights, confidence=0.95, n_sim=10000, seed=42):
    np.random.seed(seed)
    mu = returns.mean().values
    cov = returns.cov().values
    L = np.linalg.cholesky(cov)
    sim_returns = np.random.randn(n_sim, len(weights)) @ L.T + mu
    sim_portfolio = sim_returns @ weights
    var = -np.percentile(sim_portfolio, 100*(1-confidence))
    return var, sim_portfolio

var_95_mc, sim_portfolio = var_monte_carlo(returns, weights, 0.95, n_sim=20000)
var_99_mc, _ = var_monte_carlo(returns, weights, 0.99, n_sim=20000)

print("="*50)
print("📊 蒙特卡洛模拟法结果 (20000次)")
print("="*50)
print(f"95% VaR: {var_95_mc:.4f} = {var_95_mc:.2%}")
print(f"99% VaR: {var_99_mc:.4f} = {var_99_mc:.2%}")

# 收敛性分析
sim_sizes = [100, 500, 1000, 2000, 5000, 10000, 20000, 50000]
var_estimates = []
for n in sim_sizes:
    v, _ = var_monte_carlo(returns, weights, 0.95, n_sim=n, seed=n)
    var_estimates.append(v)

var_true = var_estimates[-1]
print(f"\n📈 收敛性: 最终收敛值(50000次)={var_true:.6f}")
print(f"20000次偏差: {(var_estimates[6]-var_true)/var_true*100:.2f}%")

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogx(sim_sizes, var_estimates, 'o-', color='steelblue', markersize=6)
ax.axhline(var_true, color='red', linestyle='--', alpha=0.7, label=f'收敛值={var_true:.6f}')
ax.fill_between(sim_sizes, var_true*0.99, var_true*1.01, alpha=0.2, color='green', label='±1%')
ax.set_xlabel('模拟次数(对数)'); ax.set_ylabel('95% VaR')
ax.set_title('蒙特卡洛 VaR 收敛曲线'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('var_monte_carlo.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. CVaR（条件风险价值 / Expected Shortfall）

### 为什么需要 CVaR？

VaR 只告诉你"门槛"在哪，不告诉你超过门槛后会发生什么。

- **组合 A**：95% VaR = 100 万，超过的损失都刚好 101 万
- **组合 B**：95% VaR = 100 万，但 5% 极端情况可能损失 1000 万

**VaR 无法区分这两个组合。CVaR 可以。**

### 定义

$$\text{CVaR}_\alpha = \mathbb{E}[\text{损失} \mid \text{损失} > \text{VaR}_\alpha]$$

即：超过 VaR 的损失的条件期望值。

| 特性 | VaR | CVaR |
|------|-----|------|
| 尾部信息 | ❌ 无 | ✅ 包含整个尾部 |
| 次可加性 | ❌ 不满足 | ✅ 满足（一致风险度量） |
| Basel III 推荐 | 旧标准 | ✅ **新标准** |

In [ ]:
# ========== CVaR 计算与对比 ==========
cvar_95_hs = cvar_historical(portfolio_returns, 0.95)
cvar_99_hs = cvar_historical(portfolio_returns, 0.99)
cvar_95_par = cvar_parametric(portfolio_returns, 0.95)
cvar_99_par = cvar_parametric(portfolio_returns, 0.99)

def cvar_monte_carlo(returns, weights, confidence=0.95, n_sim=10000, seed=42):
    _, sp = var_monte_carlo(returns, weights, confidence, n_sim, seed)
    v = -np.percentile(sp, 100*(1-confidence))
    exc = sp[sp <= -v]
    return -exc.mean() if len(exc) > 0 else v

cvar_95_mc = cvar_monte_carlo(returns, weights, 0.95, n_sim=20000)

comparison = pd.DataFrame({
    '指标': ['95% VaR', '95% CVaR', '99% VaR', '99% CVaR'],
    '历史模拟法': [f'{var_95_hs:.4%}', f'{cvar_95_hs:.4%}', f'{var_99_hs:.4%}', f'{cvar_99_hs:.4%}'],
    '参数法': [f'{var_95_par:.4%}', f'{cvar_95_par:.4%}', f'{var_99_par:.4%}', f'{cvar_99_par:.4%}'],
    '蒙特卡洛法': [f'{var_95_mc:.4%}', f'{cvar_95_mc:.4%}', f'{var_99_mc:.4%}', '—'],
})
print("="*70)
print("📊 VaR vs CVaR 完整对比")
print("="*70)
display(comparison)

# 可视化
fig, ax = plt.subplots(figsize=(12, 5))
n, bins, patches = ax.hist(portfolio_returns, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(-var_95_hs, color='orange', linestyle='--', linewidth=2, label=f'历史VaR 95%={var_95_hs:.2%}')
ax.axvline(-cvar_95_hs, color='red', linestyle='-', linewidth=2.5, label=f'历史CVaR 95%={cvar_95_hs:.2%}')
ax.axvspan(min(portfolio_returns), -var_95_hs, alpha=0.15, color='red', label='尾部区域(损失>VaR)')
ax.set_xlabel('日收益率'); ax.set_ylabel('频率密度')
ax.set_title('VaR vs CVaR：CVaR 包含更多尾部信息'); ax.legend(loc='upper left')
plt.tight_layout(); plt.savefig('var_vs_cvar.png', dpi=150, bbox_inches='tight'); plt.show()

diff_cvar_var = cvar_95_hs - var_95_hs
print(f"\n💡 关键洞察：")
print(f"  VaR(门槛)={var_95_hs:.2%} vs CVaR(平均尾部损失)={cvar_95_hs:.2%}")
print(f"  差值={diff_cvar_var:.2%} ← 差值越大说明尾部风险越严重（肥尾现象）")

## 7. 三种方法横向对比

In [ ]:
methods = ['历史模拟法', '参数法', '蒙特卡洛法']
var_95_vals = [var_95_hs, var_95_par, var_95_mc]
var_99_vals = [var_99_hs, var_99_par, var_99_mc]
cvar_95_vals = [cvar_95_hs, cvar_95_par, cvar_95_mc]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['steelblue', 'coral', 'seagreen']

for ax, vals, title in zip(axes, [var_95_vals, var_99_vals, cvar_95_vals],
                            ['95% VaR', '99% VaR', '95% CVaR']):
    bars = ax.bar(methods, [v*100 for v in vals], color=colors, edgecolor='white')
    ax.set_ylabel('%'); ax.set_title(title)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.2%}', ha='center', fontsize=9)

plt.tight_layout(); plt.savefig('method_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

# 差异分析
diff_par_hs = abs(var_95_par-var_95_hs)/var_95_hs*100
print(f"\n参数法 vs 历史模拟法差异: {diff_par_hs:.1f}%")
print(f"  → {'差异较大，分布偏离正态' if diff_par_hs>10 else '差异较小，接近正态'}")

print("\n💡 方法选择建议：")
print("  日常监控→历史模拟法 | 监管报告→参数法+肥尾调整 | 复杂组合→蒙特卡洛 | Basel合规→CVaR")

## 8. 回测验证：Kupiec POF 检验

VaR 模型需要通过回测来验证准确性。**Kupiec POF 检验**（Proportion of Failures）是最经典的方法。

- **原假设 H₀**：模型准确，例外率 = $1-\alpha$
- **检验统计量**：似然比统计量，服从 $\chi^2(1)$ 分布

In [ ]:
# ========== Kupiec 回测检验 ==========
def kupiec_test(returns, var_values, confidence=0.95):
    n = len(returns)
    exceptions = np.sum(returns < -var_values)
    exception_rate = exceptions / n
    expected_rate = 1 - confidence
    
    if exceptions == 0:
        lr_stat = -2*n*np.log(1-expected_rate)
    elif exceptions == n:
        lr_stat = -2*n*np.log(expected_rate)
    else:
        lr_stat = -2*(np.log((1-expected_rate)**(n-exceptions)*expected_rate**exceptions)
                     - np.log((1-exception_rate)**(n-exceptions)*exception_rate**exceptions))
    
    p_value = 1 - stats.chi2.cdf(lr_stat, df=1)
    return {
        '样本数': n, '例外天数': exceptions,
        '实际例外率': f'{exception_rate:.2%}', '期望例外率': f'{expected_rate:.2%}',
        'LR统计量': round(lr_stat, 2), 'p值': round(p_value, 4),
        '结论': '✓ 模型可接受' if p_value > 0.05 else '⚠ 模型需调整'
    }

print("="*60)
print("📊 Kupiec POF 回测检验")
print("="*60)

for conf, var_val in [(0.95, var_95_hs), (0.99, var_99_hs)]:
    r = kupiec_test(portfolio_returns, var_val, conf)
    print(f"\n{conf:.0%} VaR 回测:")
    for k, v in r.items():
        print(f"  {k}: {v}")

# 例外日可视化
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(portfolio_returns.index, portfolio_returns, color='steelblue', alpha=0.7, linewidth=0.8, label='日收益率')
ax.axhline(-var_95_hs, color='orange', linestyle='--', linewidth=1.5, label=f'95%VaR={var_95_hs:.2%}')
ax.axhline(-var_99_hs, color='red', linestyle='-', linewidth=1.5, label=f'99%VaR={var_99_hs:.2%}')

exc_95 = portfolio_returns < -var_95_hs
exc_99 = portfolio_returns < -var_99_hs
ax.scatter(portfolio_returns.index[exc_95], portfolio_returns[exc_95], color='red', s=20, zorder=5, alpha=0.8, label=f'突破95%VaR:{exc_95.sum()}天')
ax.scatter(portfolio_returns.index[exc_99], portfolio_returns[exc_99], color='darkred', s=40, marker='x', zorder=6, label=f'突破99%VaR:{exc_99.sum()}天')
ax.set_xlabel('日期'); ax.set_ylabel('日收益率'); ax.set_title('VaR 回测：例外日标注')
ax.legend(loc='lower left'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('var_backtest.png', dpi=150, bbox_inches='tight'); plt.show()

print(f"\n📋 95% VaR 合规: 期望例外率5%, 实际={exc_95.sum()/len(portfolio_returns):.2%}")
print(f"  → {'在合理范围✓ (3%-7%)' if 0.03<=exc_95.sum()/len(portfolio_returns)<=0.07 else '偏离预期⚠'}")

## 9. 小结与验收

### ✅ 验收清单

- [ ] 能手算 VaR 验证代码（分位数/z值）
- [ ] 理解 VaR 的局限性（无尾部信息、不满足次可加性）
- [ ] 能解释 CVaR 为什么更优（包含整个尾部、一致风险度量）
- [ ] 能用三种方法计算同一个组合的 VaR 并对比差异
- [ ] 能对 VaR 模型做 Kupiec 回测检验

### 🔑 核心要点

| 概念 | 一句话总结 |
|------|-----------|
| VaR | "95%置信度下，单日最多亏多少" |
| CVaR | "如果亏得比 VaR 还多，平均会亏多少" |
| 历史模拟法 | 用过去的数据直接取分位数 |
| 参数法 | 假设正态分布，用 μ+zσ 算 |
| 蒙特卡洛法 | 大量模拟，取模拟结果的分位数 |
| Kupiec 检验 | 检查实际例外率是否与期望一致 |

### 📂 生成文件

- `19_VaR_CVaR.ipynb` — 本 Notebook
- `notes/quant/19-VaR-CVaR.md` — Obsidian 精华笔记
- `var_*.png` — 4 张可视化图表

---

> **下节预告**：序号 20 — **Markowitz 均值方差优化**，构建有效前沿，找到最优组合权重。